### Part 1: What is an LLM Gateway?
Think of an LLM Gateway as a smart middleware layer that sits between your application and multiple LLM providers (OpenAI, Anthropic, Google, Groq, Cohere, local models, etc.).

                    ┌─────────────────────────────┐
                    │       Your Application      │
                    │  (Chatbot, RAG, Agent, etc) │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │       LLM GATEWAY           │
                    │  • Routing                  │
                    │  • Fallbacks                │
                    │  • Caching                  │
                    │  • Rate Limiting            │
                    │  • Cost Tracking            │
                    │  • Observability            │
                    └──────┬─────┬─────┬─────┬────┘
                           │     │     │     │
                           ▼     ▼     ▼     ▼
                        OpenAI Claude Gemini Groq
### Without a Gateway (The Pain)
- Different SDKs and APIs for every provider
- No fallback if one provider goes down
- No central place to track costs
- Hard to switch models without rewriting code
- No caching → paying twice for the same query
### With a Gateway (The Joy)
- One unified API for 100+ providers
- Automatic fallbacks if a provider fails
- Centralized logging, cost tracking, rate limiting
- Swap models with a config change, no code rewrite
- Cache repeated queries → save money

In [12]:
import warnings 
import logging 

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

# Now import Litellm normally
from litellm import completion 


In [13]:
import litellm
litellm.suppress_debug_info = True

In [14]:
import warnings
import logging

# Keep the recording clean — suppress noisy AWS-related warnings
warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [15]:
# Load API keys from a .env file
# Create a .env file in the same folder with:

import os
from dotenv import load_dotenv
load_dotenv()

# Quick check
# print("OpenAI key loaded:    ",os.getenv("OPENAI_API_KEY"))
# print("Anthropic key loaded: ",os.getenv("ANTHROPIC_API_KEY"))
# print("Groq key loaded:      ",os.getenv("GROQ_API_KEY"))

True

### The Simplest LiteLLM Example — Unified API
The biggest pain point: every provider has a different SDK.

LiteLLM gives you one function — completion() — that works with all of them. Look at how clean this is:

In [16]:
from litellm import completion

# Same code, different providers — just change the `model` string!

# Call OpenAI
from litellm import completion
import os

response = completion(
    model="gemini/gemini-2.5-flash",
    messages=[
        {"role": "user", "content": "Explain RAG in one sentence."}
    ]
)

print(response.choices[0].message.content)



# Call Groq (super fast inference)
response_groq = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("Groq:      ", response_groq.choices[0].message.content)

RAG enhances large language models by retrieving relevant external information and incorporating it as context to generate more accurate and up-to-date responses.
Groq:       RAG (Retrieval, Augmentation, Generation) is an artificial intelligence framework that combines retrieval of relevant information from a large corpus, augmentation of that information, and generation of text based on the retrieved and augmented data to produce more accurate and informative responses.


In [17]:
from litellm import completion

prompt = "Explain RAG in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("🔵 OpenAI",     "gpt-4o-mini"),
    ("🟢 Groq",       "groq/llama-3.3-70b-versatile"),
    ("🟣 Anthropic",  "claude-3-5-haiku-20241022"),
    ("🟡 Gemini",     "gemini/gemini-1.5-flash"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: ❌ {type(e).__name__}")

🔵 OpenAI       : ❌ RateLimitError
🟢 Groq         : RAG (Retrieve, Augment, Generate) is a type of artificial intelligence model tha
🟣 Anthropic    : ❌ BadRequestError
🟡 Gemini       : ❌ NotFoundError


### Automatic Fallbacks — When OpenAI Goes Down
Real story: OpenAI had a 4-hour outage in November 2023. Apps that hard-coded gpt-4 went completely dark.

With a gateway, if one provider fails, we automatically fall back to another. Production apps must have this.

In [18]:
from litellm import completion

# Define a fallback chain: try GPT first, then Claude, then Groq
response = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=["groq/llama-3.3-70b-versatile",
        "gpt-4o-mini"
        
    ]
)

print("Response:", response.choices[0].message.content[:200], "...")
print("\nWhich model actually answered?", response.model)

Task was destroyed but it is pending!
task: <Task pending name='Task-77' coro=<LoggingWorker._worker_loop() running at c:\Users\bharat singh thakur\Desktop\Python\LangChainNew\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_worker.py:111>>


Response: An LLM (Large Language Model) Gateway is an interface or a platform that enables users to interact with a large language model, such as a conversational AI or a language generator. The gateway acts as ...

Which model actually answered? llama-3.3-70b-versatile


### Cost Tracking — Know Where Your Money Goes
LiteLLM automatically calculates the cost of every call using its built-in pricing database. No more surprise bills.

In [3]:
from litellm import completion, completion_cost

response = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Write a haiku about AI."}]
)

# Get the exact USD cost of this single call
cost = completion_cost(
    completion_response=response,
    model="groq/llama-3.3-70b-versatile"
)

print("Response:    ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")

Response:     Metal mind awakes
Digital thoughts unfold slow
Future's gentle hum

Input tokens:  42
Output tokens: 15
Cost:         $0.00003663


### Part 6: Caching — Don't Pay Twice for the Same Question
If 100 users ask "What is RAG?", you don't need to call the LLM 100 times.

Enable in-memory caching with one line:

In [1]:
import litellm

# 🧹 Reset any callbacks/strategies left over from earlier cells
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

# Also clear any router-strategy state
litellm.cache = None

print("LiteLLM state reset — ready for clean caching demo")

LiteLLM state reset — ready for clean caching demo
